# PM2.5 Maroc - Feature Engineering

Objectif: preparer le dataset pour la prediction PM2.5 heure par heure sur les prochaines 24h.

Ce notebook ne fait pas encore d'entrainement ML. Il cree seulement:

- les features temporelles
- les features meteo propres
- les lags PM2.5 sans fuite temporelle
- les rolling features
- les targets futures `h1` a `h24`
- les fichiers CSV prets pour l'etape modele


## 0. Configuration

In [5]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

# Paths
INPUT_PATH = Path(r"E:\pipeline\test2\eda_outputs\cities_last_year_v2.csv")
OUTPUT_DIR = Path(r"E:\pipeline\test2\feature_engineering_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Horizons forecasting
HORIZONS_24H = list(range(1, 25))

LAGS_HOURS = [
    1,2,3,6,12,
    24,48,72,
    168,
    336
]
ROLLING_WINDOWS = [
    3,
    6,
    12,
    24,
    48,
    72,
    168
]

## 1. Charger les donnees propres de l'EDA

In [6]:
df = pd.read_csv(INPUT_PATH)

df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["city"] = df["city"].astype(str).str.strip()

df = df.dropna(subset=["datetime", "city", "pm2_5"]).copy()
df = df.sort_values(["city", "datetime"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Villes:", df["city"].nunique())
print("Periode:", df["datetime"].min(), "->", df["datetime"].max())
display(df.head())

Shape: (1315200, 20)
Villes: 50
Periode: 2023-06-09 00:00:00 -> 2026-06-08 23:00:00


,datetime,country,city,lat,lon,temperature,humidity,pressure,wind_speed,clouds,weather_main,pm2_5,pm10,co,no2,so2,o3,nh3,month,is_pm25_extreme_p99
0,2023-06-09 00:00:00,Morocco,Agadir,30.42018,-9.59815,19.0,97,1012.2,6.1,90,3,6.6,13.7,99.0,3.9,1.6,50.0,0.1,6,False
1,2023-06-09 01:00:00,Morocco,Agadir,30.42018,-9.59815,18.6,98,1011.6,3.5,100,3,7.5,13.8,99.0,4.7,1.9,47.0,0.1,6,False
2,2023-06-09 02:00:00,Morocco,Agadir,30.42018,-9.59815,18.5,98,1010.6,5.1,100,3,8.3,15.6,99.0,5.2,2.9,42.0,0.1,6,False
3,2023-06-09 03:00:00,Morocco,Agadir,30.42018,-9.59815,18.3,99,1010.2,1.9,18,0,7.9,15.6,99.0,4.6,3.0,38.0,0.2,6,False
4,2023-06-09 04:00:00,Morocco,Agadir,30.42018,-9.59815,18.5,97,1010.1,5.4,90,3,8.2,13.7,100.0,4.4,3.7,38.0,0.1,6,False


## 3. Features temporelles

In [7]:
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month
df["day"] = df["datetime"].dt.day
df["weekofyear"] = df["datetime"].dt.isocalendar().week.astype(int)

# cyclic encoding
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

display(df[["datetime", "city", "hour", "hour_sin", "hour_cos", "dayofweek", "dow_sin", "dow_cos"]].head())

,datetime,city,hour,hour_sin,hour_cos,dayofweek,dow_sin,dow_cos
0,2023-06-09 00:00:00,Agadir,0,0.000000,1.000000,4,-0.433884,-0.900969
1,2023-06-09 01:00:00,Agadir,1,0.258819,0.965926,4,-0.433884,-0.900969
2,2023-06-09 02:00:00,Agadir,2,0.500000,0.866025,4,-0.433884,-0.900969
3,2023-06-09 03:00:00,Agadir,3,0.707107,0.707107,4,-0.433884,-0.900969
4,2023-06-09 04:00:00,Agadir,4,0.866025,0.500000,4,-0.433884,-0.900969


## Moyennes historiques

In [8]:
df["pm25_city_hour_mean"] = (
    df.groupby(["city", "hour"])["pm2_5"]
      .transform(
          lambda x: x.shift(1).expanding().mean()
      )
)

In [9]:
df = df.sort_values(["city", "datetime"])

df["pm25_city_dow_mean"] = (
    df.groupby(["city", "dayofweek"])["pm2_5"]
      .transform(
          lambda x: x.shift(1).expanding().mean()
      )
)

In [10]:
df["pm25_city_month_mean"] = (
    df.groupby(["city", "month"])["pm2_5"]
      .transform(
          lambda x: x.shift(1).expanding().mean()
      )
)

## 4. Features meteo

La direction du vent est circulaire: `359 deg` est proche de `0 deg`. On la transforme donc en `sin/cos`.

In [11]:
# interaction simple utile
df["temp_change_1h"] = df.groupby("city")["temperature"].diff(1)
df["humidity_change_1h"] = df.groupby("city")["humidity"].diff(1)
df["temp_humidity"] = (
    df["temperature"] * df["humidity"]
)

df["temp_wind"] = (
    df["temperature"] * df["wind_speed"]
)

df["humidity_wind"] = (
    df["humidity"] * df["wind_speed"]
)

# smoothing pollution (très important)
df["pm25_ewm_6h"] = (
    df.groupby("city")["pm2_5"]
      .transform(lambda x:
          x.shift(1).ewm(span=6).mean()
      )
)

df["pm25_ewm_24h"] = (
    df.groupby("city")["pm2_5"]
      .transform(lambda x:
          x.shift(1).ewm(span=24).mean()
      )
)
display(df[[c for c in ["temp_change_1h", "humidity_change_1h", "pm25_change_1h", "pm25_ewm_6h", "pm25_ewm_24h"] if c in df.columns]].head())

,temp_change_1h,humidity_change_1h,pm25_ewm_6h,pm25_ewm_24h
0,NaN,NaN,NaN,NaN
1,-0.4,1.0,6.600000,6.600000
2,-0.1,0.0,7.125000,7.068750
3,-0.2,1.0,7.653211,7.513823
4,0.2,-2.0,7.748536,7.622756


## 5. Lags PM2.5 sans fuite temporelle


In [12]:
base = df[["city", "datetime", "pm2_5"]].copy()

for lag in LAGS_HOURS:
    tmp = base.copy()
    tmp["datetime"] = tmp["datetime"] + pd.to_timedelta(lag, unit="h")
    tmp = tmp.rename(columns={"pm2_5": f"pm25_lag_{lag}h"})
    df = df.merge(tmp, on=["city", "datetime"], how="left")

lag_cols = [f"pm25_lag_{l}h" for l in LAGS_HOURS]
display(df[["city", "datetime", "pm2_5"] + lag_cols].head(30))
display(df[lag_cols].notna().mean().mul(100).round(2).to_frame("available_pct"))

,city,datetime,pm2_5,pm25_lag_1h,pm25_lag_2h,pm25_lag_3h,pm25_lag_6h,pm25_lag_12h,pm25_lag_24h,pm25_lag_48h,pm25_lag_72h,pm25_lag_168h,pm25_lag_336h
0,Agadir,2023-06-09 00:00:00,6.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Agadir,2023-06-09 01:00:00,7.5,6.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Agadir,2023-06-09 02:00:00,8.3,7.5,6.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Agadir,2023-06-09 03:00:00,7.9,8.3,7.5,6.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Agadir,2023-06-09 04:00:00,8.2,7.9,8.3,7.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Agadir,2023-06-09 05:00:00,8.7,8.2,7.9,8.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Agadir,2023-06-09 06:00:00,9.2,8.7,8.2,7.9,6.6,NaN,NaN,NaN,NaN,NaN,NaN
7,Agadir,2023-06-09 07:00:00,9.1,9.2,8.7,8.2,7.5,NaN,NaN,NaN,NaN,NaN,NaN
8,Agadir,2023-06-09 08:00:00,10.0,9.1,9.2,8.7,8.3,NaN,NaN,NaN,NaN,NaN,NaN
9,Agadir,2023-06-09 09:00:00,9.8,10.0,9.1,9.2,7.9,NaN,NaN,NaN,NaN,NaN,NaN


,available_pct
pm25_lag_1h,100.00
pm25_lag_2h,99.99
pm25_lag_3h,99.99
pm25_lag_6h,99.98
pm25_lag_12h,99.95
pm25_lag_24h,99.91
pm25_lag_48h,99.82
pm25_lag_72h,99.73
pm25_lag_168h,99.36
pm25_lag_336h,98.72


In [13]:
df["pm25_change_1h"] = df.groupby("city")["pm2_5"].diff(1)
df["pm25_change_3h"] = (
    df["pm2_5"] - df["pm25_lag_3h"]
)

df["pm25_change_6h"] = (
    df["pm2_5"] - df["pm25_lag_6h"]
)

df["pm25_change_24h"] = (
    df["pm2_5"] - df["pm25_lag_24h"]
)

## 6. Rolling features sans tricher sur les trous horaires

On cree une grille horaire complete par ville temporairement. Les rolling features sont calculees seulement avec les observations passees.

In [14]:
rolling_frames = []

for city, g in df.groupby("city"):
    g = g.sort_values("datetime").copy()

    full_index = pd.date_range(g["datetime"].min(), g["datetime"].max(), freq="h")

    hourly = g.set_index("datetime")[["pm2_5"]].reindex(full_index)
    hourly["city"] = city

    shifted = hourly["pm2_5"].shift(1)

    for w in ROLLING_WINDOWS:
        hourly[f"roll_mean_{w}h"] = shifted.rolling(w).mean()
        hourly[f"roll_std_{w}h"] = shifted.rolling(w).std()
        hourly[f"roll_min_{w}h"] = shifted.rolling(w).min()
        hourly[f"roll_max_{w}h"] = shifted.rolling(w).max()

    hourly = hourly.reset_index().rename(columns={"index": "datetime"})

    rolling_frames.append(hourly)

roll_df = pd.concat(rolling_frames, ignore_index=True)


df = df.merge(roll_df, on=["city", "datetime", "pm2_5"], how="left")

roll_mean_cols = [c for c in df.columns if "roll_mean_" in c]

cols_to_show = [c for c in ["city", "datetime", "pm2_5"] if c in df.columns] + roll_mean_cols

display(df[cols_to_show].head(30))
display(df[roll_mean_cols].notna().mean().mul(100).round(2).to_frame("available_pct"))

,city,datetime,pm2_5,roll_mean_3h,roll_mean_6h,roll_mean_12h,roll_mean_24h,roll_mean_48h,roll_mean_72h,roll_mean_168h
0,Agadir,2023-06-09 00:00:00,6.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Agadir,2023-06-09 01:00:00,7.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Agadir,2023-06-09 02:00:00,8.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Agadir,2023-06-09 03:00:00,7.9,7.466667,NaN,NaN,NaN,NaN,NaN,NaN
4,Agadir,2023-06-09 04:00:00,8.2,7.900000,NaN,NaN,NaN,NaN,NaN,NaN
5,Agadir,2023-06-09 05:00:00,8.7,8.133333,NaN,NaN,NaN,NaN,NaN,NaN
6,Agadir,2023-06-09 06:00:00,9.2,8.266667,7.866667,NaN,NaN,NaN,NaN,NaN
7,Agadir,2023-06-09 07:00:00,9.1,8.700000,8.300000,NaN,NaN,NaN,NaN,NaN
8,Agadir,2023-06-09 08:00:00,10.0,9.000000,8.566667,NaN,NaN,NaN,NaN,NaN
9,Agadir,2023-06-09 09:00:00,9.8,9.433333,8.850000,NaN,NaN,NaN,NaN,NaN


,available_pct
roll_mean_3h,99.99
roll_mean_6h,99.98
roll_mean_12h,99.95
roll_mean_24h,99.91
roll_mean_48h,99.82
roll_mean_72h,99.73
roll_mean_168h,99.36


## 7. TARGETS MULTI-HORIZON (3 SCALAIRES)


In [15]:
def create_targets(df, horizons, prefix):
    base = df[["city", "datetime", "pm2_5"]].copy()
    base = base.rename(columns={"pm2_5": "target"})

    for h in horizons:
        tmp = base.copy()
        tmp["datetime"] = tmp["datetime"] - pd.to_timedelta(h, unit="h")
        tmp = tmp.rename(columns={"target": f"{prefix}_h{h}"})
        df = df.merge(tmp, on=["city", "datetime"], how="left")

    return df


df = create_targets(df, HORIZONS_24H, "target24")

target_cols = []

target_cols += [f"target24_h{h}" for h in HORIZONS_24H]

target_cols = sorted(list(set(target_cols)))

display(df.head(10))

,datetime,country,city,lat,lon,temperature,humidity,pressure,wind_speed,clouds,weather_main,pm2_5,pm10,co,no2,so2,o3,nh3,month,is_pm25_extreme_p99,hour,dayofweek,day,weekofyear,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,pm25_city_hour_mean,pm25_city_dow_mean,pm25_city_month_mean,temp_change_1h,humidity_change_1h,temp_humidity,temp_wind,humidity_wind,pm25_ewm_6h,pm25_ewm_24h,pm25_lag_1h,pm25_lag_2h,pm25_lag_3h,pm25_lag_6h,pm25_lag_12h,pm25_lag_24h,pm25_lag_48h,pm25_lag_72h,pm25_lag_168h,pm25_lag_336h,pm25_change_1h,pm25_change_3h,pm25_change_6h,pm25_change_24h,roll_mean_3h,roll_std_3h,roll_min_3h,roll_max_3h,roll_mean_6h,roll_std_6h,roll_min_6h,roll_max_6h,roll_mean_12h,roll_std_12h,roll_min_12h,roll_max_12h,roll_mean_24h,roll_std_24h,roll_min_24h,roll_max_24h,roll_mean_48h,roll_std_48h,roll_min_48h,roll_max_48h,roll_mean_72h,roll_std_72h,roll_min_72h,roll_max_72h,roll_mean_168h,roll_std_168h,roll_min_168h,roll_max_168h,target24_h1,target24_h2,target24_h3,target24_h4,target24_h5,target24_h6,target24_h7,target24_h8,target24_h9,target24_h10,target24_h11,target24_h12,target24_h13,target24_h14,target24_h15,target24_h16,target24_h17,target24_h18,target24_h19,target24_h20,target24_h21,target24_h22,target24_h23,target24_h24
0,2023-06-09 00:00:00,Morocco,Agadir,30.42018,-9.59815,19.0,97,1012.2,6.1,90,3,6.6,13.7,99.0,3.9,1.6,50.0,0.1,6,False,0,4,9,23,0.000000,1.000000e+00,-0.433884,-0.900969,1.224647e-16,-1.0,NaN,NaN,NaN,NaN,NaN,1843.0,115.90,591.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.5,8.3,7.9,8.2,8.7,9.2,9.1,10.0,9.8,9.5,9.2,7.7,6.4,6.7,6.3,5.7,5.0,4.7,4.9,5.4,5.7,5.9,5.9,6.0
1,2023-06-09 01:00:00,Morocco,Agadir,30.42018,-9.59815,18.6,98,1011.6,3.5,100,3,7.5,13.8,99.0,4.7,1.9,47.0,0.1,6,False,1,4,9,23,0.258819,9.659258e-01,-0.433884,-0.900969,1.224647e-16,-1.0,NaN,6.600000,6.600000,-0.4,1.0,1822.8,65.10,343.0,6.600000,6.600000,6.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.3,7.9,8.2,8.7,9.2,9.1,10.0,9.8,9.5,9.2,7.7,6.4,6.7,6.3,5.7,5.0,4.7,4.9,5.4,5.7,5.9,5.9,6.0,6.2
2,2023-06-09 02:00:00,Morocco,Agadir,30.42018,-9.59815,18.5,98,1010.6,5.1,100,3,8.3,15.6,99.0,5.2,2.9,42.0,0.1,6,False,2,4,9,23,0.500000,8.660254e-01,-0.433884,-0.900969,1.224647e-16,-1.0,NaN,7.050000,7.050000,-0.1,0.0,1813.0,94.35,499.8,7.125000,7.068750,7.5,6.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.9,8.2,8.7,9.2,9.1,10.0,9.8,9.5,9.2,7.7,6.4,6.7,6.3,5.7,5.0,4.7,4.9,5.4,5.7,5.9,5.9,6.0,6.2,6.5
3,2023-06-09 03:00:00,Morocco,Agadir,30.42018,-9.59815,18.3,99,1010.2,1.9,18,0,7.9,15.6,99.0,4.6,3.0,38.0,0.2,6,False,3,4,9,23,0.707107,7.071068e-01,-0.433884,-0.900969,1.224647e-16,-1.0,NaN,7.466667,7.466667,-0.2,1.0,1811.7,34.77,188.1,7.653211,7.513823,8.3,7.5,6.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.4,1.3,NaN,NaN,7.466667,0.850490,6.6,8.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.2,8.7,9.2,9.1,10.0,9.8,9.5,9.2,7.7,6.4,6.7,6.3,5.7,5.0,4.7,4.9,5.4,5.7,5.9,5.9,6.0,6.2,6.5,7.0
4,2023-06-09 04:00:00,Morocco,Agadir,30.42018,-9.59815,18.5,97,1010.1,5.4,90,3,8.2,13.7,100.0,4.4,3.7,38.0,0.1,6,False,4,4,9,23,0.866025,5.000000e-01,-0.433884,-0.900969,1.224647e-16,-1.0,NaN,7.575000,7.575000,0.2,-2.0,1794.5,99.90,523.8,7.748536,7.622756,7.9,8.3,7.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3,0.7,NaN,NaN,7.900000,0.400000,7.5,8.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.7,9.2,9.1,10.0,9.8,9.5,9.2,7.7,6.4,6.7,6.3,5.7,5.0,4.7,4.9,5.4,5.7,5.9,5.9,6.0,6.2,6.5,7.0,7.0
5,2023-06-09 05:00:00,Morocco,Agadir,30.42018,-9.59815,18.3,96,1010.0,6.6,17,0,8.7,12.8,101.0,4.9,4.5,36.0,0.1,6,False,5,4,9,23,0.965926,2.588190e-01

## 8. Flags qualite

In [16]:
df["is_extreme"] = df["pm2_5"] > df["pm2_5"].quantile(0.99)
df["is_zero"] = df["pm2_5"].eq(0)

for lag in LAGS_HOURS:
    df[f"has_lag_{lag}"] = df[f"pm25_lag_{lag}h"].notna()

print("PM2.5 p99 extreme rows:", int(df["is_extreme"].sum()))
print("PM2.5 zero rows:", int(df["is_zero"].sum()))
display(df[["city", "datetime", "pm2_5", "is_extreme", "is_zero"]].head())

PM2.5 p99 extreme rows: 13101
PM2.5 zero rows: 0


,city,datetime,pm2_5,is_extreme,is_zero
0,Agadir,2023-06-09 00:00:00,6.6,False,False
1,Agadir,2023-06-09 01:00:00,7.5,False,False
2,Agadir,2023-06-09 02:00:00,8.3,False,False
3,Agadir,2023-06-09 03:00:00,7.9,False,False
4,Agadir,2023-06-09 04:00:00,8.2,False,False


## 9. Audit final

In [17]:
print("Rows:", len(df))
print("Cities:", df["city"].nunique())
print("Period:", df["datetime"].min(), "->", df["datetime"].max())

print("\nRows by city:")
display(df.groupby("city", observed=True).size().sort_values(ascending=False).to_frame("rows"))

print("\nLag availability:")
display(df[lag_cols].notna().mean().mul(100).round(2).to_frame("available_pct"))

print("\nTarget availability h1-h24 + h48 + h72:")
display(df[target_cols].notna().mean().mul(100).round(2).to_frame("available_pct"))

print("\nMissing values top 30:")
display(df.isna().sum().sort_values(ascending=False).head(30).to_frame("missing"))


Rows: 1315200
Cities: 50
Period: 2023-06-09 00:00:00 -> 2026-06-08 23:00:00

Rows by city:


,rows
city,
Agadir,26304
Ait Melloul,26304
Al Fqih Ben Çalah,26304
Al Hoceïma,26304
Ben Guerir,26304
Beni Mellal,26304
Berkane,26304
Berrechid,26304
Bouskoura,26304



Lag availability:


,available_pct
pm25_lag_1h,100.00
pm25_lag_2h,99.99
pm25_lag_3h,99.99
pm25_lag_6h,99.98
pm25_lag_12h,99.95
pm25_lag_24h,99.91
pm25_lag_48h,99.82
pm25_lag_72h,99.73
pm25_lag_168h,99.36
pm25_lag_336h,98.72



Target availability h1-h24 + h48 + h72:


,available_pct
target24_h1,100.00
target24_h10,99.96
target24_h11,99.96
target24_h12,99.95
target24_h13,99.95
target24_h14,99.95
target24_h15,99.94
target24_h16,99.94
target24_h17,99.94
target24_h18,99.93



Missing values top 30:


,missing
pm25_lag_336h,16800
pm25_lag_168h,8400
roll_max_168h,8400
roll_min_168h,8400
roll_std_168h,8400
roll_mean_168h,8400
roll_max_72h,3600
roll_min_72h,3600
roll_std_72h,3600
roll_mean_72h,3600


## 10. Colonnes utilisables plus tard pour le modele


In [18]:
safe_features = [
    "city", "lat", "lon",
    "hour", "dayofweek", "month", "day", "weekofyear",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
]

safe_features += lag_cols
safe_features += [c for c in df.columns if "roll_" in c]
safe_features += [
    "temp_change_1h",
    "humidity_change_1h",
    "pm25_change_1h",
    "pm25_ewm_6h",
    "pm25_ewm_24h"
]

safe_features += [

    "pm25_city_hour_mean",
    "pm25_city_dow_mean",
    "pm25_city_month_mean",

    "temp_humidity",
    "temp_wind",
    "humidity_wind",

    "pm25_change_3h",
    "pm25_change_6h",
    "pm25_change_24h"
]

safe_features = [c for c in safe_features if c in df.columns]

print("Nombre de features safe:", len(safe_features))
display(pd.DataFrame({"safe_feature": safe_features}))

Nombre de features safe: 66


,safe_feature
0,city
1,lat
2,lon
3,hour
4,dayofweek
...,...
61,temp_wind
62,humidity_wind
63,pm25_change_3h
64,pm25_change_6h


## 11. Sauvegarde

In [19]:
base_features_path = OUTPUT_DIR / "pm25_features_base.csv"
supervised_path = OUTPUT_DIR / "pm25_supervised_h1_h24.csv"
safe_features_path = OUTPUT_DIR / "safe_feature_columns.txt"
audit_path = OUTPUT_DIR / "feature_engineering_audit.md"

df.to_csv(base_features_path, index=False)

# Dataset avec au moins une target future disponible
df_supervised = df.dropna(subset=target_cols, how="all").copy()
df_supervised.to_csv(supervised_path, index=False)

safe_features_path.write_text("\n".join(safe_features), encoding="utf-8")

audit_text = f"""# Feature Engineering Audit

- Rows: {len(df)}
- Cities: {df['city'].nunique()}
- Period: {df['datetime'].min()} -> {df['datetime'].max()}

- Base features file: {base_features_path}
- Supervised file: {supervised_path}
- Safe features file: {safe_features_path}

- PM2.5 extreme p99 rows: {int(df['is_extreme'].sum())}
- PM2.5 zero rows: {int(df['is_zero'].sum())}

## Lag availability (%)
{df[lag_cols].notna().mean().mul(100).round(2).to_string()}

## Target availability (%)
{df[target_cols].notna().mean().mul(100).round(2).to_string()}
"""

audit_path.write_text(audit_text, encoding="utf-8")

print("Saved:", base_features_path)
print("Saved:", supervised_path)
print("Saved:", safe_features_path)
print("Saved:", audit_path)

Saved: E:\pipeline\test2\feature_engineering_outputs\pm25_features_base.csv
Saved: E:\pipeline\test2\feature_engineering_outputs\pm25_supervised_h1_h24.csv
Saved: E:\pipeline\test2\feature_engineering_outputs\safe_feature_columns.txt
Saved: E:\pipeline\test2\feature_engineering_outputs\feature_engineering_audit.md
